In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## DATASET IMPORT

In [ ]:
submission=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')
train=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

## DESCIPTIVE STATISTICS

In [ ]:
train.describe()

In [ ]:
print(train.dtypes)
train.columns

In [ ]:
if 'SalePrice' not in test.columns:
    test['SalePrice']=0
df=pd.concat([train,test],axis=0)
print(df.head())

In [ ]:
df.head()

You can actually combine the train and test dataset as the manipulations of the columns in train and test will be done together.

In [ ]:
df.isnull().sum()

In [ ]:
print(df.isna().sum()>0)

In [ ]:
df_null=df[df.isnull().sum()[df.isnull().sum()>0].index]
print(df_null)

In [ ]:
df.isnull().sum()[df.isnull().sum()>0]

In [ ]:
sns.heatmap(df_null.isnull())

In [ ]:
df_objects=df[df.select_dtypes(include=['object']).columns]
df=df.drop(df[df_objects.isna().sum()[df_objects.isna().sum()>1100].index],axis=1)

In [ ]:
df_objects=df_objects.drop(df_objects[df_objects.isna().sum()[df_objects.isna().sum()>1100].index],axis=1)
df_objects=df_objects.fillna('null')
df_objects_encoded=pd.get_dummies(df_objects)

In [ ]:
df_objects_encoded.head()

In [ ]:
df_objects.head()

In [ ]:
df_objects['Utilities'].value_counts()

In [ ]:
for i in df_objects_encoded.columns:
    if 'null' in i:
        df_objects_encoded= df_objects_encoded.drop(i,axis=1)
        print(i)

In [ ]:
new_df=pd.concat([df,df_objects_encoded],axis=1)

In [ ]:
new_df=new_df.drop(new_df.select_dtypes(include=['object']).columns, axis=1)
new_df.isna().sum()[new_df.isna().sum()>0]

In [ ]:
import numpy as np

# Columns where missing values will be filled with MODE
Mode_columns = ['GarageCars', 'GarageYrBlt', 'BsmtFullBath', 'BsmtHalfBath']

# Columns where missing values will be filled with MEAN
Mean_columns = [
    'LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
    'BsmtUnfSF', 'TotalBsmtSF', 'GarageArea'
]

# Fill missing values with mode
for i in Mode_columns:
    new_df[i] = new_df[i].fillna(new_df[i].mode()[0])

# Fill missing values with mean (rounded)
for i in Mean_columns:
    new_df[i] = new_df[i].fillna(np.round(new_df[i].mean()))

# Check if any missing values remain
new_df.isna().sum()[new_df.isna().sum() > 0]

In [ ]:
print(new_df.columns[:20])

In [ ]:
print(new_df.columns[20:40])

In [ ]:
print(new_df.columns[40:60])

In [ ]:
training_df=new_df[0:len(train)]
testing_df=new_df[len(train):]
testing_df.drop(columns=['SalePrice'],inplace=True,errors="ignore")
print(testing_df.shape)
print(training_df.shape)

## EDA

In [ ]:
plt.bar(training_df['YearBuilt'],training_df['SalePrice'])

In [ ]:
plt.scatter(training_df['LotArea'],training_df['SalePrice'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, VotingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
!pip install xgboost
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
X=training_df.drop(columns=['SalePrice'])
y=training_df['SalePrice']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
model_1=LinearRegression()
model_1.fit(X_train,y_train)
y_1=model_1.predict(X_test)
score_1_r2=r2_score(y_1,y_test)
score_1_mse=mean_squared_error(y_1,y_test)
print(score_1_r2)
print(score_1_mse)

In [ ]:
model_2=RandomForestRegressor()
model_2.fit(X_train,y_train)
y_2=model_2.predict(X_test)
score_2_r2=r2_score(y_2,y_test)
score_2_mse=mean_squared_error(y_2,y_test)
print(score_2_r2)
print(score_2_mse)

In [ ]:
model_3=BaggingRegressor()
model_3.fit(X_train,y_train)
y_3=model_3.predict(X_test)
score_3_r2=r2_score(y_3,y_test)
score_3_mse=mean_squared_error(y_3,y_test)
print(score_3_r2)
print(score_3_mse)

In [ ]:
model_4=KNeighborsRegressor()
model_4.fit(X_train,y_train)
y_4=model_4.predict(X_test)
score_4_r2=r2_score(y_4,y_test)
score_4_mse=mean_squared_error(y_4,y_test)
print(score_4_r2)
print(score_4_mse)

In [ ]:
model_5=AdaBoostRegressor()
model_5.fit(X_train,y_train)
y_5=model_5.predict(X_test)
score_5_r2=r2_score(y_5,y_test)
score_5_mse=mean_squared_error(y_5,y_test)
print(score_5_r2)
print(score_5_mse)

In [ ]:
model_6=DecisionTreeRegressor()
model_6.fit(X_train,y_train)
y_6=model_1.predict(X_test)
score_6_r2=r2_score(y_6,y_test)
score_6_mse=mean_squared_error(y_6,y_test)
print(score_6_r2)
print(score_6_mse)

In [ ]:
models=['LinearRegression','RandomForestRegressor','BaggingRegressor','KNeighborsRegressor','AdaBoostRegressor','DecisionTreeRegressor']
scores_r2=[score_1_r2,score_2_r2,score_3_r2,score_4_r2,score_5_r2,score_6_r2]
scores_mse=[score_1_mse,score_2_mse,score_3_mse,score_4_mse,score_5_mse,score_6_mse]
plt.xticks(rotation=30)
plt.bar(models,scores_r2)
plt.title('R2_score')

In [ ]:
plt.xticks(rotation=30)
plt.bar(models,scores_mse)
plt.title('MSE_score')

# SO THE BEST MODEL IS RANDOM FOREST REGRESSOR
## Low MSE and High R2_Score

In [ ]:
sort_indices = np.argsort(y_test)
# Plot
plt.figure(figsize=(10, 6))
plt.plot(np.arange(len(y_test)), y_test, label='Original')
plt.plot(np.arange(len(y_2)), y_2, label='Predicted')
plt.xlabel('Index')
plt.ylabel('Price')
plt.title('Original vs Predicted Prices')
plt.legend()
plt.show()

In [ ]:
pred=model_2.predict(testing_df)
final=pd.DataFrame()
submission=pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')
final['Id']=submission['Id']
final['SalePrice']=pred
final.to_csv('/kaggle/working/output.csv',index=False)